# Ollama + LangChain + Pydantic で構造化出力デモ（v3）
タスク: **カスタマーサポート問い合わせのトリアージ**

v2 の結果: スキーマ強制(B,C)では `urgency` がほぼ常に1になった。原因の仮説は
「Ollama は `response_format` のスキーマを文法制約にだけ使い、**プロンプトには入れない**ため、`Field(description)` の基準をモデルが見ていない」。
v3 ではスキーマ全文をプロンプトにも入れる条件(B2, D)を足して検証する。さらに `reasoning` の上限を150→300字にした（v2では文の途中で打ち切られていた）。

（以下は v2 の改善点）
1. 比較を公平に: スキーマなし側にも「JSONで出力」「スキーマ全文」をプロンプトで与え、コードフェンス除去も行う
2. `urgency` の判断基準を `Field(description=...)` に明記し、`reasoning`（理由）をスキーマの**先頭**に置く
3. 正解ラベルつきの評価セルを追加し、3条件を数値で比較する

| 条件 | スキーマ強制 | 基準の説明 | reasoning先出し |
| --- | --- | --- | --- |
| A. プロンプトのみ | なし | あり | あり |
| B. スキーマ強制（簡易） | あり | なし | なし |
| B2. スキーマ強制（簡易）＋プロンプトにもスキーマ | あり | なし | なし |
| C. スキーマ強制＋基準＋reasoning | あり | あり（モデルには見えない） | あり |
| D. C ＋プロンプトにもスキーマ | あり | あり（プロンプトで見える） | あり |

※ ランタイムは GPU (T4) 推奨。

In [1]:
# 1. セットアップ
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
!pip -q install langchain-openai pydantic

In [2]:
# 2. Ollamaサーバ起動 + モデル取得
import subprocess, time, requests

MODEL = "qwen3.5:9b"

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434"); break
    except Exception:
        time.sleep(1)

!ollama pull {MODEL} 2>&1 | tail -n 2   # 進捗の大量出力を抑える
!ollama -v

writing manifest 
success 
ollama version is 0.40.14


In [3]:
# 3. スキーマ定義
from enum import Enum
from pydantic import BaseModel, Field


class Category(str, Enum):
    billing = "billing"
    bug = "bug"
    account = "account"
    feature_request = "feature_request"
    other = "other"


class Sentiment(str, Enum):
    positive = "positive"
    neutral = "neutral"
    negative = "negative"


# --- B: 簡易版（説明なし・reasoningなし） ---
class TicketTriageBasic(BaseModel):
    category: Category
    urgency: int = Field(ge=1, le=5)
    sentiment: Sentiment
    summary: str = Field(max_length=80)
    actions: list[str] = Field(min_length=1, max_length=3)
    needs_escalation: bool


# --- C: 基準つき＋reasoningを先頭に置く（理由を書いてから数値を決めさせる） ---
class TicketTriageResult(BaseModel):
    reasoning: str = Field(
        max_length=300,
        description="緊急度とエスカレーション要否を判断した理由（日本語・300文字以内）。結論から簡潔に書き、他の項目より先に書く",
    )
    category: Category = Field(
        description="billing=請求/支払い, bug=不具合/障害, account=ログイン/アカウント, feature_request=要望, other=それ以外",
    )
    urgency: int = Field(
        ge=1, le=5,
        description=(
            "緊急度の基準: 5=全顧客に影響するサービス停止/情報漏えい, 4=一部顧客の業務停止/至急の金銭トラブル, "
            "3=業務に支障があるが回避策あり, 2=軽微な不具合/問い合わせ, 1=要望/感謝/雑談"
        ),
    )
    sentiment: Sentiment = Field(description="顧客の感情")
    summary: str = Field(max_length=80, description="問い合わせの要約（日本語・80文字以内）")
    actions: list[str] = Field(min_length=1, max_length=3, description="サポート担当が取るべき次のアクション")
    needs_escalation: bool = Field(
        description="開発/上位担当への引き継ぎが必要か。urgencyが4以上、またはサポート単独で解決できない不具合ならtrue",
    )

print(list(TicketTriageResult.model_json_schema()["properties"]))  # 並び順の確認

['reasoning', 'category', 'urgency', 'sentiment', 'summary', 'actions', 'needs_escalation']


In [4]:
# 4. LLMクライアントと推論関数
import json, re
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

SYSTEM_BASE = "あなたはSaaSのカスタマーサポート責任者です。問い合わせ文を読み、トリアージしてください。"

def build_llm(schema: type[BaseModel] | None = None) -> ChatOpenAI:
    kwargs = {}
    if schema is not None:  # スキーマ強制（Ollamaの構造化出力）
        kwargs["model_kwargs"] = {
            "response_format": {
                "type": "json_schema",
                "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()},
            }
        }
    return ChatOpenAI(
        model=MODEL,
        base_url="http://localhost:11434/v1",
        api_key="ollama",
        temperature=0,
        seed=42,
        max_tokens=600,
        max_retries=3,
        reasoning_effort="none",  # thinking無効化（vLLMのenable_thinking=False相当）
        **kwargs,
    )

def parse_json(text: str, schema: type[BaseModel]):
    # コードフェンスや前後の説明文を除去して、最初の{〜最後の}を取り出す（全条件で共通）
    m = re.search(r"\{.*\}", text, re.S)
    return schema.model_validate_json(m.group(0) if m else text)

def triage(text: str, schema=TicketTriageResult, force_schema=True, inject=False):
    system = SYSTEM_BASE
    if inject or not force_schema:  # スキーマ全文をプロンプトにも入れる（A, B2, D）
        system += "\n出力は次のJSON Schemaに従うJSONのみ。説明文は不要。\n" + json.dumps(
            schema.model_json_schema(), ensure_ascii=False)
    llm = build_llm(schema if force_schema else None)
    res = llm.invoke([SystemMessage(content=system), HumanMessage(content=text)])
    return parse_json(res.content, schema)

In [5]:
# 5. 動作確認（条件C）
DEMO = "【至急】本番環境のAPIが全リクエストで500を返しています。顧客対応が完全に止まっています。"
print(triage(DEMO).model_dump_json(indent=2))

{
  "reasoning": "この問い合わせは「本番環境の API が全リクエストで 500 エラーを返し、顧客対応が完全に停止している」という記述から、システム全体の障害が発生しており、ビジネスに重大な影響（収益損失、顧客離れ）を与えていることが明確です。これは単なる機能不具合や問い合わせ処理の遅延ではなく、即座に解決が必要な「P1（最優先）」レベルのインシデントです。カスタマーサポート責任者として、まずは技術チームへの即時通知と、顧客への謝罪および状況説明の準備を最優先事項と判断します。",
  "category": "bug",
  "urgency": 1,
  "sentiment": "negative",
  "summary": "本番環境 API 全件 500 エラーによるサービス停止（P1 インシデント）",
  "actions": [
    "技術サポートチーム（SRE/開発者）に即時アラートを送信し、障害対応を要請する",
    "ステータスページや SNS などで顧客への状況報告を開始する",
    "顧客担当者に謝罪文と復旧までの見込み時間を伝えるためのテンプレートを用意する"
  ],
  "needs_escalation": true
}


## 6. 正解ラベルつき評価

In [6]:
# 6. 評価データ: (問い合わせ, 正解カテゴリ, 許容する緊急度の範囲, エスカレーション正解 or None=判定対象外)
DATASET = [
    ("昨日から管理画面にログインできません。パスワードリセットのメールも届かず、明日の朝までに月次レポートを出す必要があり困っています。", "account", (3, 4), None),
    ("今月の請求額が先月の2倍になっています。プランは変更していないはずです。確認をお願いします。", "billing", (2, 3), None),
    ("CSVエクスポートで日本語が文字化けします。Excelで開くと全部記号になります。", "bug", (2, 3), False),
    ("ダッシュボードにダークモードがあると嬉しいです。いつも使いやすくて助かっています！", "feature_request", (1, 1), False),
    ("【至急】本番環境のAPIが全リクエストで500を返しています。顧客対応が完全に止まっています。", "bug", (5, 5), True),
    ("他社のユーザーのデータが自分の画面に表示されています。情報漏えいではないでしょうか。", "bug", (5, 5), True),
    ("領収書の宛名を株式会社の正式名称に変更してほしいです。急ぎではありません。", "billing", (1, 2), False),
    ("モバイルアプリのプッシュ通知が、たまに数分遅れて届きます。", "bug", (2, 3), False),
    ("いつもありがとうございます。先日のサポート対応が丁寧で助かりました。", "other", (1, 1), False),
    ("クレジットカードで二重に課金されました。至急返金してください。", "billing", (3, 4), None),
]

def evaluate(name, schema, force_schema, inject=False, n_repeat=2):
    n = ok_parse = ok_cat = ok_urg = 0
    esc_n = ok_esc = 0
    bad = []
    for text, cat, (lo, hi), esc in DATASET * n_repeat:
        n += 1
        try:
            r = triage(text, schema, force_schema, inject)
        except Exception as e:
            bad.append((text[:20], f"PARSE_ERROR {str(e)[:60]}"))
            continue
        ok_parse += 1
        ok_cat += r.category.value == cat
        urg_ok = lo <= r.urgency <= hi
        ok_urg += urg_ok
        if esc is not None:
            esc_n += 1
            ok_esc += r.needs_escalation == esc
        if not urg_ok:
            bad.append((text[:20], f"urgency={r.urgency} (期待 {lo}-{hi})"))
    return {
        "条件": name, "パース成功": f"{ok_parse}/{n}", "カテゴリ一致": f"{ok_cat}/{n}",
        "緊急度が範囲内": f"{ok_urg}/{n}", "エスカレ一致": f"{ok_esc}/{esc_n}", "_bad": bad,
    }

results = [
    evaluate("A. プロンプトのみ（基準+reasoning）", TicketTriageResult, force_schema=False),
    evaluate("B. スキーマ強制（簡易）", TicketTriageBasic, force_schema=True),
    evaluate("B2. スキーマ強制（簡易）＋プロンプトにもスキーマ", TicketTriageBasic, force_schema=True, inject=True),
    evaluate("C. スキーマ強制＋基準+reasoning", TicketTriageResult, force_schema=True),
    evaluate("D. C＋プロンプトにもスキーマ", TicketTriageResult, force_schema=True, inject=True),
]

In [7]:
# 7. 結果表示
import pandas as pd
df = pd.DataFrame([{k: v for k, v in r.items() if k != "_bad"} for r in results])
display(df)

for r in results:
    print(f"\n[{r['条件']}] 外れたケース（先頭5件）")
    for t, msg in r["_bad"][:5]:
        print(f"  - {t}... : {msg}")

,条件,パース成功,カテゴリ一致,緊急度が範囲内,エスカレ一致
0,A. プロンプトのみ（基準+reasoning）,20/20,20/20,18/20,12/14
1,B. スキーマ強制（簡易）,20/20,20/20,10/20,12/14
2,B2. スキーマ強制（簡易）＋プロンプトにもスキーマ,20/20,20/20,16/20,14/14
3,C. スキーマ強制＋基準+reasoning,20/20,18/20,8/20,10/14
4,D. C＋プロンプトにもスキーマ,20/20,20/20,18/20,12/14



[A. プロンプトのみ（基準+reasoning）] 外れたケース（先頭5件）
  - 今月の請求額が先月の2倍になっています。... : urgency=4 (期待 2-3)
  - 今月の請求額が先月の2倍になっています。... : urgency=4 (期待 2-3)

[B. スキーマ強制（簡易）] 外れたケース（先頭5件）
  - 昨日から管理画面にログインできません。パ... : urgency=1 (期待 3-4)
  - 今月の請求額が先月の2倍になっています。... : urgency=1 (期待 2-3)
  - 【至急】本番環境のAPIが全リクエストで... : urgency=1 (期待 5-5)
  - 他社のユーザーのデータが自分の画面に表示... : urgency=1 (期待 5-5)
  - クレジットカードで二重に課金されました。... : urgency=1 (期待 3-4)

[B2. スキーマ強制（簡易）＋プロンプトにもスキーマ] 外れたケース（先頭5件）
  - 昨日から管理画面にログインできません。パ... : urgency=5 (期待 3-4)
  - クレジットカードで二重に課金されました。... : urgency=5 (期待 3-4)
  - 昨日から管理画面にログインできません。パ... : urgency=5 (期待 3-4)
  - クレジットカードで二重に課金されました。... : urgency=5 (期待 3-4)

[C. スキーマ強制＋基準+reasoning] 外れたケース（先頭5件）
  - 昨日から管理画面にログインできません。パ... : urgency=1 (期待 3-4)
  - CSVエクスポートで日本語が文字化けしま... : urgency=1 (期待 2-3)
  - ダッシュボードにダークモードがあると嬉し... : urgency=2 (期待 1-1)
  - 【至急】本番環境のAPIが全リクエストで... : urgency=1 (期待 5-5)
  - 他社のユーザーのデータが自分の画面に表示... : urgency=1 (期待 5-5)

[D. C＋プロンプトにもスキーマ] 外れたケース（先頭5件）
  - 今月の請求額が先月の2倍になっています。... : 

## 読み方のメモ
- **A vs C**: 同じ情報をプロンプトで渡すだけでも成功するか、スキーマ強制が必要かを見る。
- **B vs B2 / C vs D**: プロンプトにスキーマを入れるだけで `urgency` が改善するか（v2の仮説の検証）。
- **B vs C**: スキーマをプロンプトに入れない場合、`description` の基準は効かない（モデルに見えない）はず。
- スキーマ強制は形式を保証するが、`max_length` や `ge/le` などの値の制約・内容の妥当性までは保証しない。評価データでの確認が必要。
- 評価データは10件×2回と小さいので、差が数件なら誤差の範囲。件数を増やすか、`n_repeat` を上げて確認する。
- `seed` 固定でも Ollama では結果が揺れることがある。